# The guardrail lifecycle

This offline lab runs the internal HR/IT support assistant through define, observe, decide, execute, recover, measure, and improve.

## 0. Setup

The fixtures contain synthetic identities, tools, and labels only, so the lab needs no credentials or network access. Detector scores are frozen fixture data, not a live model; this makes every metric reproducible.

In [ ]:
import json
import sys
from dataclasses import asdict
from pathlib import Path

LESSON_DIR = Path.cwd()
sys.path.insert(0, str(LESSON_DIR))
import lifecycle_lab as lab

fixtures = LESSON_DIR / 'fixtures'
policy_v1 = lab.Policy.from_dict(lab.load_json(fixtures / 'policy_v1.json'))
policy_v2 = lab.Policy.from_dict(lab.load_json(fixtures / 'policy_v2.json'))
traffic = [lab.TrafficItem(**item) for item in lab.load_json(fixtures / 'traffic.json')]
incident = lab.TrafficItem(**lab.load_json(fixtures / 'incident.json'))
by_id = {item.request_id: item for item in traffic}
print(len(traffic), policy_v1.version, policy_v2.version, incident.request_id)

## 1. Define policy

A policy is versioned data with an owner, threshold, prohibited and required behaviors, escalation conditions, and tool rules. Version 2 keeps the detector threshold unchanged and adds an authorization rule at the payroll tool boundary.

In [ ]:
for policy in (policy_v1, policy_v2):
    print({
        'policy_id': policy.policy_id,
        'version': policy.version,
        'owner': policy.owner,
        'threshold': policy.threshold,
        'mode': policy.mode.value,
        'tool_rules': policy.tool_rules,
    })

## 2. Map the trust boundaries

Each boundary gets the control that protects its data or authority transition. A detector at the input boundary cannot grant permission at the tool gateway.

In [ ]:
boundaries = [
    ('identity', 'authentication and business-unit scope'),
    ('retrieval', 'document authorization and provenance'),
    ('model context', 'separate instructions from untrusted data'),
    ('tool gateway', 'allowlist, role, schema, and approval'),
    ('memory', 'retention and tenant isolation'),
    ('human approval', 'checkpoint consequential actions'),
    ('logs and datasets', 'hash identity and minimize sensitive content'),
]
for boundary, control in boundaries:
    print(f'{boundary}: {control}')

## 3. Observe

Observation keeps the minimum signal needed for a decision: role, business unit, tool name, score, detector version, and policy version. The raw user ID is replaced with a SHA-256 identity hash, so the trace remains useful without becoming a second data breach.

In [ ]:
observation = lab.observe(by_id['attack-03'], policy_v1)
print(asdict(observation))
assert 'user-a03' not in json.dumps(asdict(observation))

## 4. Decide

The lifecycle checks deterministic tool authorization before applying the detector threshold. A score-driven decision carries confidence; a deterministic role denial has `confidence=None`.

In [ ]:
for policy in (policy_v1, policy_v2):
    record = lab.decide(lab.observe(incident, policy), policy)
    print(policy.version, json.loads(record.to_json()))

## 5. Constrain, execute, and verify

A permitted read tool can execute with a receipt, while the same idempotency key prevents duplicate side effects on retry. Verification checks the receipt after execution rather than assuming a transport response proves business success.

In [ ]:
ledger = lab.Ledger()
call = {'name': 'read_ticket', 'arguments': {'ticket_id': 'ticket-42'}}
first = ledger.execute_with_receipt(call, 'request-42')
second = ledger.execute_with_receipt(call, 'request-42')
print(first)
print('same receipt:', first == second, 'verified:', ledger.verify(first), 'entries:', len(ledger.entries))

## 6. Recover

Recovery is a policy action, not another typed decision: blocked requests get a compliant alternative and escalations pause for review. Only a `retryable_error` may retry, and the cap forces escalation when repeated attempts do not resolve the problem.

In [ ]:
retry_record = lab.DecisionRecord(
    request_id='retry-01', decision=lab.Decision.BLOCK,
    reason_codes=['retryable_error'], policy_version=1,
    confidence=None, would_be_decision=lab.Decision.BLOCK,
    next_step='offer a policy-compliant alternative',
)
print(lab.recover(retry_record, attempt=0))
print(lab.recover(retry_record, attempt=2))
print(lab.recover(retry_record, attempt=0, max_retries=0))

## 7. Measure

The same frozen traffic can be evaluated in shadow, alert, and enforce modes to separate decisions that would have happened from decisions actually applied. False positives and friction include boundary items as legitimate, while blocked attack attempts and unsafe completions remain separate counts.

In [ ]:
for mode in (lab.Mode.SHADOW, lab.Mode.ALERT, lab.Mode.ENFORCE):
    metrics, records = lab.evaluate(traffic, policy_v1, mode)
    print(mode.value, asdict(metrics), 'first:', records[0].decision.value, records[0].would_be_decision.value)
print('would-be enforce metrics:', asdict(lab.evaluate_would_be(traffic, policy_v1)))

## 8. Improve

The incident is an attack that v1 misses because its score is below the unchanged threshold. Version 2 catches it with the narrowest responsible fix—a role rule at `issue_payroll_adjustment`—and the metric delta shows the result.

In [ ]:
v1_metrics, v1_records = lab.evaluate([incident], policy_v1, lab.Mode.ENFORCE)
v2_metrics, v2_records = lab.evaluate([incident], policy_v2, lab.Mode.ENFORCE)
print('v1:', v1_records[0].decision.value, v1_records[0].reason_codes)
print('v2:', v2_records[0].decision.value, v2_records[0].reason_codes)
traffic_v1, _ = lab.evaluate(traffic, policy_v1, lab.Mode.ENFORCE)
traffic_v2, _ = lab.evaluate(traffic, policy_v2, lab.Mode.ENFORCE)
print('traffic delta v2-v1:', lab.compare(traffic_v1, traffic_v2))

## Exercises

1. Add a policy rule for an unlisted risk and predict its shadow, alert, and enforce outcomes.
2. Change the threshold without changing the tool rule and predict which metric moves first.
3. Add the incident to a regression fixture and explain why the narrowest fix belongs at the tool boundary.